In [ ]:
import numpy as np
import pandas as pd

# =====================================================
# Print data after each Radix-2 FFT stage and butterfly
# FFT 8-point example
# =====================================================

x = np.array([1, 2, 3, 4, 5, 6, 7, 8], dtype=complex)
N = len(x)

def bit_reverse_indices(N):
    bits = int(np.log2(N))
    return [int(format(i, f"0{bits}b")[::-1], 2) for i in range(N)]

def fmt(z):
    z = np.round(z, 4)
    if abs(z.imag) < 1e-9:
        return str(int(z.real)) if z.real == int(z.real) else str(z.real)
    if abs(z.real) < 1e-9:
        return f"{z.imag}j"
    sign = "+" if z.imag >= 0 else "-"
    return f"{z.real}{sign}{abs(z.imag)}j"

# Bit reversal
bit_rev = bit_reverse_indices(N)
x_stage = x[bit_rev].copy()

print("Input:")
print([fmt(v) for v in x])

print("\nBit-reversal indices:")
print(bit_rev)

print("\nAfter bit-reversal:")
print([fmt(v) for v in x_stage])

records = []

num_stages = int(np.log2(N))

for stage in range(1, num_stages + 1):
    m = 2 ** stage
    half_m = m // 2
    W_m = np.exp(-2j * np.pi / m)

    print(f"\n==============================")
    print(f"Stage {stage}: group size = {m}")
    print(f"==============================")

    butterfly_count = 1

    for k in range(0, N, m):
        for j in range(half_m):
            index_a = k + j
            index_b = k + j + half_m

            A = x_stage[index_a]
            B = x_stage[index_b]
            W = W_m ** j
            WB = W * B

            upper_output = A + WB
            lower_output = A - WB

            print(f"\nButterfly {butterfly_count}")
            print(f"Input indices: ({index_a}, {index_b})")
            print(f"A = {fmt(A)}")
            print(f"B = {fmt(B)}")
            print(f"W = {fmt(W)}")
            print(f"W × B = {fmt(WB)}")
            print(f"Upper output = A + W×B = {fmt(upper_output)}")
            print(f"Lower output = A - W×B = {fmt(lower_output)}")

            records.append({
                "Stage": f"Stage {stage}",
                "Butterfly": butterfly_count,
                "Input Indices": f"({index_a}, {index_b})",
                "A": fmt(A),
                "B": fmt(B),
                "Twiddle W": fmt(W),
                "W × B": fmt(WB),
                "Upper Output": fmt(upper_output),
                "Lower Output": fmt(lower_output)
            })

            x_stage[index_a] = upper_output
            x_stage[index_b] = lower_output

            butterfly_count += 1

    print(f"\nOutput after Stage {stage}:")
    print([fmt(v) for v in x_stage])

df_butterfly = pd.DataFrame(records)

print("\nFinal FFT Output:")
print([fmt(v) for v in x_stage])

print("\nNumPy FFT Output:")
print([fmt(v) for v in np.fft.fft(x)])

df_butterfly

Input:
['1', '2', '3', '4', '5', '6', '7', '8']

Bit-reversal indices:
[0, 4, 2, 6, 1, 5, 3, 7]

After bit-reversal:
['1', '5', '3', '7', '2', '6', '4', '8']

Stage 1: group size = 2

Butterfly 1
Input indices: (0, 1)
A = 1
B = 5
W = 1
W × B = 5
Upper output = A + W×B = 6
Lower output = A - W×B = -4

Butterfly 2
Input indices: (2, 3)
A = 3
B = 7
W = 1
W × B = 7
Upper output = A + W×B = 10
Lower output = A - W×B = -4

Butterfly 3
Input indices: (4, 5)
A = 2
B = 6
W = 1
W × B = 6
Upper output = A + W×B = 8
Lower output = A - W×B = -4

Butterfly 4
Input indices: (6, 7)
A = 4
B = 8
W = 1
W × B = 8
Upper output = A + W×B = 12
Lower output = A - W×B = -4

Output after Stage 1:
['6', '-4', '10', '-4', '8', '-4', '12', '-4']

Stage 2: group size = 4

Butterfly 1
Input indices: (0, 2)
A = 6
B = 10
W = 1
W × B = 10
Upper output = A + W×B = 16
Lower output = A - W×B = -4

Butterfly 2
Input indices: (1, 3)
A = -4
B = -4
W = -1.0j
W × B = 4.0j
Upper output = A + W×B = -4.0+4.0j
Lower output = A - W

,Stage,Butterfly,Input Indices,A,B,Twiddle W,W × B,Upper Output,Lower Output
0,Stage 1,1,"(0, 1)",1,5,1,5,6,-4
1,Stage 1,2,"(2, 3)",3,7,1,7,10,-4
2,Stage 1,3,"(4, 5)",2,6,1,6,8,-4
3,Stage 1,4,"(6, 7)",4,8,1,8,12,-4
4,Stage 2,1,"(0, 2)",6,10,1,10,16,-4
5,Stage 2,2,"(1, 3)",-4,-4,-1.0j,4.0j,-4.0+4.0j,-4.0-4.0j
6,Stage 2,3,"(4, 6)",8,12,1,12,20,-4
7,Stage 2,4,"(5, 7)",-4,-4,-1.0j,4.0j,-4.0+4.0j,-4.0-4.0j
8,Stage 3,1,"(0, 4)",16,20,1,20,36,-4
9,Stage 3,2,"(1, 5)",-4.0+4.0j,-4.0+4.0j,0.7071-0.7071j,5.6569j,-4.0+9.6569j,-4.0-1.6569j
